Cargar archivos json

In [ ]:
import json
import os

POLYVORE_PATH = "./polyvore_kaggle/polyvore_outfits/"

# Cargamos los  archivos json con la informacion de los diferentes outfits
def cargar_outfit_splits(base_dir=POLYVORE_PATH):
    with open(base_dir + "disjoint/" + "train.json") as f:
        train = json.load(f)
    with open(base_dir + "disjoint/" + "valid.json") as f:
        validation = json.load(f)
    with open(base_dir + "disjoint/" + "test.json") as f:
        test = json.load(f)
    with open(base_dir + "polyvore_item_metadata.json") as f:
        item_metadata = json.load(f)
    with open(base_dir + "polyvore_outfit_titles.json") as f:
        outfit_titles = json.load(f)
    return train, validation, test, item_metadata, outfit_titles

train, validation, test, item_metadata, outfit_titles = cargar_outfit_splits()

### Visualizar un item aleatorio

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2

def mostrar_imagen_aleatoria():
    images_dir = POLYVORE_PATH + "images/"
    image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]
    img_path = os.path.join(images_dir, random.choice(image_files))
    
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.imshow(img)
    plt.axis("off")
    plt.title(img_path)
    plt.show()

mostrar_imagen_aleatoria()

### Mostrar items y ourfits por particion

In [ ]:
def contar_items_unicos(split):
    unique_items = set()
    for outfit in split:
        for item in outfit['items']:
            unique_items.add(item['item_id'])
    return len(unique_items)

print(f" Conjunto      ||   Numero de Outfits      ||    Numero de items únicos     ")
print(f"==========================================================================")
print(f"ENTRENAMIENTO  ||      {len(train)}               ||   {contar_items_unicos(train)}")
print(f"VALIDACION     ||      {len(validation)}                ||   {contar_items_unicos(validation)}")
print(f"PRUEBA         ||      {len(test)}               ||   {contar_items_unicos(test)}")

In [ ]:
import json
with open(POLYVORE_PATH + 'polyvore_item_metadata.json') as f:
    meta = json.load(f)
semantic_cats = {v.get('semantic_category') for v in meta.values()}
print(len(semantic_cats), semantic_cats)

### Mostrar un outfit aleatorio

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
import os

def mostrar_outfit_aleatorio(split):
    outfit = random.choice(split)
    items = sorted(outfit['items'], key=lambda x: x['index'])
    n = len(items)

    fig, axes = plt.subplots(1, n, figsize=(3 * n, 4))
    if n == 1:
        axes = [axes]

    fig.suptitle(f"Outfit ID: {outfit['set_id']} TITLE: {outfit_titles.get(outfit['set_id']).get('url_name')}", fontsize=12)

    for ax, item in zip(axes, items):
        img_path = os.path.join(POLYVORE_PATH + "images/", item['item_id'] + ".jpg")
        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "No image", ha="center", va="center")
        ax.set_title(f"#{item['index']}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

mostrar_outfit_aleatorio(train)

### Mostrar distribucion de items por outfit

In [ ]:
import matplotlib.pyplot as plt

def plot_items_por_outfit(split, split_name="Split"):
    items_count = [len(outfit['items']) for outfit in split]

    fig, ax = plt.subplots(figsize=(10, 5))
    min_items = min(items_count)
    max_items = max(items_count)
    bins = range(min_items, max_items + 2)

    counts, edges, bars = ax.hist(items_count, bins=bins, align='left', rwidth=0.8, color='steelblue', edgecolor='white')
    
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                    str(int(count)), ha='center', va='bottom', fontsize=9)

    ax.set_xlabel("Número de items por outfit")
    ax.set_ylabel("Número de outfits")
    ax.set_title(f"Distribución de items por outfit — {split_name}")
    ax.set_xticks(range(min_items, max_items + 1))

    plt.tight_layout()
    plt.show()

plot_items_por_outfit(train, "Train")
# plot_items_por_outfit(validation, "Val")
# plot_items_por_outfit(test, "Test")

In [ ]:
plot_items_por_outfit(test, "Test")

### Mostrar shapes de las imagenes por outfits: ¿son todas las imagenes 300x300x3?

In [ ]:
import os
import cv2
from collections import Counter

def mostrar_shapes_imagenes(split):
    shapes = []
    
    for outfit in split:
        for item in outfit['items']:
            img_path = os.path.join(POLYVORE_PATH + "images/", item['item_id'] + ".jpg")
            if os.path.exists(img_path):
                img = cv2.imread(img_path)
                if img is not None:
                    h, w, c = img.shape[:]
                    shapes.append((h, w, c))
    
    shape_counts = Counter(shapes)
    print(f"Shapes únicas: {len(shape_counts)}")
    print(f"\n{'Shape (H x W X C)':<20} {'Cantidad':>10} {'%':>8}")
    print("-" * 40)
    for shape, count in shape_counts.most_common():
        pct = count / len(shapes) * 100
        print(f"{str(shape):<20} {count:>10} {pct:>7.1f}%")

print('SHAPES CONJUNTO ENTRENAMIENTO')
mostrar_shapes_imagenes(train)

print('SHAPES CONJUNTO VALIDACION')
mostrar_shapes_imagenes(validation)

print('SHAPES CONJUNTO PRUEBA')
mostrar_shapes_imagenes(test)

### N/As

In [ ]:
# Como se observa en los histogramas, no existes conjuntos sin items
# Vamos a explorar si existen conjuntos con ids de items o de sets vacios

def contar_ids_faltantes(split):
    missing_outfit_id = 0
    missing_item_id = 0
    missing_any = 0

    for outfit in split:
        outfit_missing = False

        if outfit['set_id'] not in outfit_titles:
            missing_outfit_id += 1
            outfit_missing = True

        for item in outfit['items']:
            if item['item_id'] not in item_metadata:
                missing_item_id += 1
                if not outfit_missing:
                    outfit_missing = True

        if outfit_missing:
            missing_any += 1

    total = len(split)
    print(f"Total outfits:                        {total}")
    print(f"Outfits sin set_id en outfit_titles:  {missing_outfit_id} ({missing_outfit_id/total*100:.1f}%)")
    print(f"Items sin item_id en item_metadata:   {missing_item_id} (se cuentan ids faltantes no cantidad de items)")
  
    print(f"Outfits con algún dato faltante:      {missing_any} ({missing_any/total*100:.1f}%)")
print("CONJUNTO DE ENTRENAMIENTO")
contar_ids_faltantes(train)
print("\nCONJUNTO DE VALIDACION")
contar_ids_faltantes(validation)
print("\nCONJUNTO DE PRUEBA")
contar_ids_faltantes(test)

In [ ]:
# Tomar el primer item y ver sus keys
first_id = next(iter(item_metadata))
print(item_metadata[first_id].keys())

# Verificar si todos los items tienen exactamente los mismos atributos
all_keys = [tuple(v.keys()) for v in item_metadata.values()]
unique_key_sets = set(all_keys)
print(unique_key_sets)

item_metadata["211444470"]

In [ ]:
def contar_nas_por_atributo(item_metadata):
    from collections import defaultdict
    
    na_values = {"", None, "N/A", "n/a", "null"}
    counts = defaultdict(int)
    na_item_ids = set()
    total = len(item_metadata)
    
    for item_id, item in item_metadata.items():
        for key, value in item.items():
            if isinstance(value, list):
                if len(value) == 0:
                    counts[key] += 1
                    na_item_ids.add(item_id)
            elif value in na_values:
                counts[key] += 1
                na_item_ids.add(item_id)
    
    print(f"Total items: {total}")
    print(f"Items con algún NA: {len(na_item_ids)} ({len(na_item_ids)/total*100:.1f}%)")
    print(f"\n{'Atributo':<25} {'NAs':>8} {'%':>8}")
    print("-" * 43)
    for key, count in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"{key:<25} {count:>8} {count/total*100:>7.1f}%")

    return na_item_ids

na_item_ids = contar_nas_por_atributo(item_metadata)
outfits_sin_titulo = contar_nas_por_atributo(outfit_titles)

In [ ]:
def contar_outfits_con_items_en_lista(split, item_ids, split_name="Split"):
    item_ids = set(item_ids)  # asegurar lookup O(1)
    total = len(split)
    outfits_with_items = sum(
        1 for outfit in split
        if any(item['item_id'] in item_ids for item in outfit['items'])
    )
    print(f"{split_name}: {outfits_with_items}/{total} outfits ({outfits_with_items/total*100:.1f}%) contienen items con NAs")
    return outfits_with_items

contar_outfits_con_items_en_lista(train, na_item_ids, "Train")
contar_outfits_con_items_en_lista(validation, na_item_ids, "Validation")
contar_outfits_con_items_en_lista(test, na_item_ids, "Test")

### Mostrar cantidad de categorias

In [ ]:
import pandas as pd

df = pd.read_csv(POLYVORE_PATH + "categories.csv")

print(f"Categorías únicas (category_id): {df['category_id'].nunique()}")
print(f"Subcategorías únicas (sub_category): {df['sub_category'].nunique()}")
print(f"Categorías principales únicas (main_category): {df['main_category'].nunique()}")

In [ ]:
# Cargar categorías
categories = pd.read_csv(POLYVORE_PATH + "categories.csv")

def category_distribution(split, item_metadata, categories, split_name="Split"):
    # Extraer todos los item_ids del split
    item_ids = [item['item_id'] for outfit in split for item in outfit['items']]
    
    # Obtener category_id de cada item
    cat_ids = []
    for item_id in item_ids:
        if item_id in item_metadata:
            cat_id = item_metadata[item_id].get('category_id', None)
            if cat_id:
                cat_ids.append(int(cat_id))
    
    # Crear dataframe y cruzar con categorías
    df = pd.DataFrame({'category_id': cat_ids})
    df = df.merge(categories, on='category_id', how='left')
    
    print(f"\n{'='*50}")
    print(f" {split_name} — {len(item_ids)} items totales")
    print(f"{'='*50}")
    
    print("\n--- Por category_id ---")
    print(df.groupby('category_id').size().sort_values(ascending=False).to_string())
    
    print("\n--- Por sub_category ---")
    print(df.groupby('sub_category').size().sort_values(ascending=False).to_string())
    
    print("\n--- Por main_category ---")
    print(df.groupby('main_category').size().sort_values(ascending=False).to_string())

In [ ]:
category_distribution(train, item_metadata, categories, "Train")

In [ ]:
category_distribution(validation,   item_metadata, categories, "Val")

In [ ]:
category_distribution(test,  item_metadata, categories, "Test")